# Model Playground

Use this notebook to load a trained checkpoint and generate conditional samples with minimal setup.

**How to use**
1. Set `CKPT_DIR` (and optionally `EPOCH`, `N_STEPS`) in the **Configuration** cell.
2. Run all cells in order. The last cells generate samples and optionally run a quick evaluation.

## 1. Setup

Run once to set project root and import dependencies.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import jax.numpy as jnp
import flax.nnx as nnx
from omegaconf import OmegaConf
from hydra.utils import instantiate

import rootutils

rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True, cwd=True)

from src.model import *
from src.utils import *
from src.data import *
from src.eval import *
from src.factory import *
from src.probability_path import *
from src.manifold import *
from src.utils.data_util import prepare_promoter_data

seed = 0
torch.manual_seed(seed)
np.random.seed(seed)

## 2. Configuration

Set your checkpoint directory and sampling options. Change these and re-run from the next cell.

In [ ]:
# Checkpoint: directory that contains .hydra/config.yaml and ckpts/
CKPT_DIR = Path("...").resolve()

# Epoch to load (folder name under ckpts/ e.g. 0110 -> ckpts/0110/default)
EPOCH = "..."

# Number of flow steps for sampling (more steps often better quality, slower)
N_STEPS = 8

# Batch size for demo (number of sequences to generate per run)
BATCH_SIZE = 4

## 3. Load model and data

Load config, build manifold and model, restore checkpoint, and prepare the validation dataloader for conditioning signals.

In [ ]:
ckpt_path = CKPT_DIR / "ckpts" / EPOCH / "default"
config_path = CKPT_DIR / ".hydra" / "config.yaml"

if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

cfg = OmegaConf.load(config_path)
manifold = instantiate(cfg.manifold)
prior = UniformPrior(manifold, nnx.Rngs(seed)())
x_shape = tuple(cfg.x_shape)

# Model and optimizer (optimizer state is required for checkpoint restore)
model = init_flow_map_model(cfg.model, manifold, x_shape, seed=cfg.seed)
optimizer = init_optim(cfg.optim, model, 1000)
model, optimizer, epoch = restore_ckpt(ckpt_path, model, optimizer)
model.model = nnx.jit(model.model)

print(f"Loaded checkpoint from epoch {epoch}")
print(f"Manifold: {manifold}, x_shape: {x_shape}")

In [ ]:
# Validation dataloader for conditioning signals
cfg.data.val.split = "valid"
val_dataset = instantiate(cfg.data.val)
val_dataloader = DataLoader(
    dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

## 4. Generate samples

Sample one batch from the prior, condition on validation signals, and run the flow for `N_STEPS` steps. Output is on the simplex (probability per position).

In [ ]:
data_iter = iter(val_dataloader)
x_1_batch, signal_batch = next(data_iter)
x_1_batch, signal_batch = prepare_promoter_data(x_1_batch, signal_batch)
cond = {"signal": signal_batch}
x_0_batch = prior.sample(*x_1_batch.shape)

# Generate: x_0 -> x_1 on manifold, then convert to simplex
x1 = manifold.projx(model.sample_x1_with_n_steps(x_0_batch, N_STEPS, **cond))
x1_simplex = sphere_to_simplex(x1)

print(f"Generated shape: {x1_simplex.shape} (batch, length, vocab)")
print(f"Steps used: {N_STEPS}")